# 03 — Dense embeddings (Qwen3-Embedding-0.6B)

Same dense pipeline as `02`, but with the larger **Qwen3-Embedding-0.6B** (1024-dim) served via **sentence-transformers** (torch) — stronger semantic recall.

**Requires** `pip install -e '.[sentence-transformers]'` (already in the `.venv-li` kernel). The embedder differs from bge-small, so this method uses its **own** index (its own cache dir).

> The index cell embeds 300+ chunks with a 0.6B model on CPU — it can take a few minutes. Add `--gpu` to the index command if you have a CUDA torch.

### Index (slower — bigger model)

In [ ]:
import sys
# Index sample_repo for this method (CLI does the heavy ingestion wiring).
# --no-inspect = read source statically (don't import the package).
!{sys.executable} -m pydocs_mcp --config configs/dense_qwen3.yaml index sample_repo \
    --cache-dir .pydocs-cache/dense_qwen3 --force --no-inspect 2>&1 \
    | grep -E "Project:|Done:|rror" || true

### Search (Python pipeline)

In [ ]:
from nb_helpers import make_searcher, load_queries, show_results

# Build the retrieval PIPELINE in Python for this method (no CLI search).
search = make_searcher("configs/dense_qwen3.yaml", cache_dir='.pydocs-cache/dense_qwen3')
queries = load_queries()

In [ ]:
for q in queries:
    hits = await search(q['query'], limit=5)
    show_results(q, hits)

### Takeaway
A larger embedder usually sharpens ranking on subtle, paraphrased queries vs. bge-small — at a higher index/compute cost.